In [1]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import numpy as np
import json
import gseapy as gp

In [2]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"

# Loading

In [3]:
def load(d):
    with open(f"../output/{d}/leiden_results/result_communities_selected.pkl", "rb") as f:
        communities = pickle.load(f)
    with open(f"../output/{d}/leiden_results/result_communities_HGNC_selected.pkl", "rb") as f:
        communities_HGNC = pickle.load(f)
    # with open(f"../output/{d}/leiden_results/result_graph.pkl", "rb") as f:
    #     graph = pickle.load(f)    
    with open(f"../output/{d}/gene_to_index_distinct.json", "r") as file:
        gene_to_index_distinct = json.load(file)
        
    return communities,communities_HGNC,gene_to_index_distinct

In [4]:
communities_selected,communities_HGNC_selected,gene_to_index_distinct = load(DISEASE)

# index to HGNC

In [5]:
index_to_gene_distinct = {v: u for (u,v) in gene_to_index_distinct.items()}

In [6]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

# Select top genes

In [7]:
# comm_idx = 0

In [8]:
def zscore(values):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr

    mean = arr.mean()
    std = arr.std(ddof=0)

    if std == 0 or np.isnan(std):
        # no variation: all z-scores = 0
        return np.zeros_like(arr)

    return (arr - mean) / std

def community_central_genes_ranked(G, community_nodes, weight="weight"):
    C = set(community_nodes)
    H = G.subgraph(C).copy()                       # induced subgraph
    # within-community (weighted) degree
    k = {u: H.degree(u, weight=weight) for u in H}
    ks = np.array(list(k.values()), dtype=float)
    zscore_list = zscore(ks)
    Z = dict(zip(H,zscore_list))        # within-module degree z-score
    
    return Z

In [9]:
index_to_gene_distinct

{10: '1',
 11: '2',
 12: '3',
 13: '9',
 14: '10',
 15: '12',
 16: '13',
 17: '14',
 18: '15',
 19: '16',
 20: '18',
 21: '19',
 22: '20',
 23: '21',
 24: '22',
 25: '23',
 26: '24',
 27: '25',
 28: '26',
 29: '27',
 30: '28',
 31: '29',
 32: '30',
 33: '31',
 34: '32',
 35: '33',
 36: '34',
 37: '35',
 38: '36',
 39: '37',
 40: '38',
 41: '39',
 42: '40',
 43: '41',
 44: '43',
 45: '47',
 46: '48',
 47: '49',
 48: '50',
 49: '51',
 50: '52',
 51: '53',
 52: '54',
 53: '55',
 54: '56',
 55: '58',
 56: '59',
 57: '60',
 58: '69',
 59: '70',
 60: '71',
 61: '72',
 62: '81',
 63: '83',
 64: '86',
 65: '87',
 66: '88',
 67: '89',
 68: '90',
 69: '91',
 70: '92',
 71: '93',
 72: '94',
 73: '95',
 74: '97',
 75: '98',
 76: '100',
 77: '101',
 78: '102',
 79: '103',
 80: '104',
 81: '105',
 82: '107',
 83: '108',
 84: '109',
 85: '111',
 86: '112',
 87: '113',
 88: '114',
 89: '115',
 90: '116',
 91: '117',
 92: '118',
 93: '119',
 94: '120',
 95: '123',
 96: '124',
 97: '125',
 98: '126',
 9

In [10]:
# zscore_dict = community_central_genes_ranked(graph,communities[comm_idx])

In [11]:
# zscore_dict = dict(sorted(zscore_dict.items(), key=lambda x: x[1], reverse=True))

In [12]:
# zscore_dict

In [13]:
# top_genes = [u for u,v in zscore_dict.items()][:3]

In [14]:
# top_genes

# Community deepdive

In [25]:
comm_idx = 2

In [26]:
comm_HGNC = communities_HGNC_selected[comm_idx]

In [27]:
len(comm_HGNC)

1270

In [28]:
important_terms = pd.read_csv(DISEASE_FOLDER + "important_terms.csv")

In [29]:
go_df_filtered = important_terms[important_terms["Community Index"] == comm_idx]

In [30]:
go_df_filtered = go_df_filtered.sort_values(by = ["Adjusted P-value"], ascending = [True])

In [31]:
go_df_filtered

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
97,2,1270,Golgi Membrane (GO:0000139),78/427,2.377760e-15,['cellular anatomical structure'],GO_Cellular_Component_2023,1.595812e-17,0.0,0.0,3.446371,133.293797,GALNT12;SCARB2;GPSM1;GALNT11;GALNT14;ST6GALNAC...,GO:0000139,{'GO:0110165'},0.182670,NaN
103,2,1270,Bounding Membrane Of Organelle (GO:0098588),117/819,5.552438e-15,['cellular anatomical structure'],GO_Cellular_Component_2023,6.373014e-17,0.0,0.0,2.605955,97.180964,SCARB2;GALNT12;GALNT11;GALNT14;KDELR1;PITPNB;S...,GO:0098588,{'GO:0110165'},0.142857,NaN
30,2,1270,RNA Binding (GO:0003723),171/1411,3.366901e-14,['binding'],GO_Molecular_Function_2023,5.163958e-17,0.0,0.0,2.194656,82.304539,OTUD4;TCERG1;RPL31;CISD2;NOC2L;RPL8;MKI67;RRP8...,GO:0003723,{'GO:0005488'},0.121191,NaN
64,2,1270,Protein Glycosylation (GO:0006486),42/147,1.584063e-13,[],GO_Biological_Process_2023,4.623650e-17,0.0,0.0,6.066775,228.188174,GALNT12;GALNT11;GALNT14;ST6GALNAC2;GALNT18;PLO...,GO:0006486,set(),0.285714,NaN
23,2,1270,Metabolism Of RNA R-HSA-8953854,96/666,2.479166e-11,['Metabolism of RNA'],Reactome_2022,2.662906e-14,0.0,0.0,2.605218,81.430719,TUT7;RPL31;DDX47;TUT4;SPPL2A;DDX42;CWC25;YBX1;...,NaN,NaN,0.144144,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,2,1270,cullin-RING Ubiquitin Ligase Complex (GO:0031461),26/174,6.679910e-04,['protein-containing complex'],GO_Cellular_Component_2023,4.034845e-05,0.0,0.0,2.624120,26.550736,ANAPC13;DET1;PEF1;CCNF;KLHL12;KLHL13;DDA1;DCAF...,GO:0031461,{'GO:0032991'},0.149425,NaN
41,2,1270,Glycosaminoglycan biosynthesis,13/53,7.100978e-04,[],KEGG_2021_Human,1.981668e-05,0.0,0.0,4.832339,52.329333,HS3ST3B1;GLCE;CSGALNACT2;EXTL2;FUT8;CHST11;DSE...,NaN,NaN,0.245283,NaN
42,2,1270,Glycosphingolipid biosynthesis,12/45,7.100978e-04,[],KEGG_2021_Human,1.652113e-05,0.0,0.0,5.404538,59.508669,B3GALNT1;ST8SIA1;B3GALT4;B3GNT3;B3GNT2;B4GALNT...,NaN,NaN,0.266667,NaN
29,2,1270,Protein Ubiquitination (GO:0016567),52/434,7.427047e-04,['cellular process'],GO_Biological_Process_2023,7.804252e-06,0.0,0.0,2.050602,24.116808,DET1;UBE3C;UBE3D;CCNF;MAGEF1;DDA1;RNF115;UBE2Q...,GO:0016567,{'GO:0009987'},0.119816,NaN


In [32]:
go_df_filtered[go_df_filtered["Gene_set"] == "Reactome_2022"]

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
23,2,1270,Metabolism Of RNA R-HSA-8953854,96/666,2.479166e-11,['Metabolism of RNA'],Reactome_2022,2.662906e-14,0.0,0.0,2.605218,81.430719,TUT7;RPL31;DDX47;TUT4;SPPL2A;DDX42;CWC25;YBX1;...,NaN,NaN,0.144144,NaN
18,2,1270,Post-translational Protein Modification R-HSA-...,154/1383,1.217164e-09,['Metabolism of proteins'],Reactome_2022,2.614745e-12,0.0,0.0,1.965022,52.406862,RAB3B;GALNT12;GALNT11;GALNT14;KDELR1;GALNT18;C...,NaN,NaN,0.111352,NaN
26,2,1270,Metabolism Of Proteins R-HSA-392499,193/1890,2.962559e-09,['Metabolism of proteins'],Reactome_2022,9.546376e-12,0.0,0.0,1.798668,45.640937,KDELR1;RPL31;CCNF;RPL8;TFG;LONP2;FBXO6;FBXO9;S...,NaN,NaN,0.102116,NaN
25,2,1270,Membrane Trafficking R-HSA-199991,78/599,2.498259e-07,['Vesicle-mediated transport'],Reactome_2022,1.073366e-09,0.0,0.0,2.287003,47.232251,ITSN2;SCARB2;DENND1B;SCOC;C2CD5;KDELR1;DENND1A...,NaN,NaN,0.130217,NaN
19,2,1270,Vesicle-mediated Transport R-HSA-5653656,78/637,3.186330e-06,['Vesicle-mediated transport'],Reactome_2022,1.711241e-08,0.0,0.0,2.127088,38.039690,ITSN2;SCARB2;DENND1B;SCOC;C2CD5;KDELR1;DENND1A...,NaN,NaN,0.122449,NaN
48,2,1270,Processing Of Capped Intron-Containing Pre-mRN...,40/242,3.692972e-06,['Metabolism of RNA'],Reactome_2022,2.380003e-08,0.0,0.0,2.982854,52.359770,FYTTD1;NUP205;POLDIP3;SF3B6;SRSF1;DDX42;CWC25;...,NaN,NaN,0.165289,NaN
39,2,1270,Intra-Golgi And Retrograde Golgi-to-ER Traffic...,33/181,4.772621e-06,['Vesicle-mediated transport'],Reactome_2022,3.588436e-08,0.0,0.0,3.349461,57.419697,SCOC;NBAS;KDELR1;UBE2Z;BICD1;BICD2;AGPAT3;STX1...,NaN,NaN,0.182320,NaN
46,2,1270,mRNA 3-End Processing R-HSA-72187,17/58,7.613316e-06,['Metabolism of RNA'],Reactome_2022,6.542054e-08,0.0,0.0,6.184435,102.305587,FYTTD1;CHTOP;CPSF7;CPSF6;CPSF1;POLDIP3;SRSF1;U...,NaN,NaN,0.293103,NaN
21,2,1270,Asparagine N-linked Glycosylation R-HSA-446203,43/282,8.278916e-06,['Metabolism of proteins'],Reactome_2022,8.003248e-08,0.0,0.0,2.711355,44.305801,SEC23A;UBXN1;ST6GALNAC2;KDELR1;PRKCSH;UAP1;NEU...,NaN,NaN,0.152482,NaN
47,2,1270,rRNA Modification In Nucleus And Cytosol R-HSA...,17/60,1.004998e-05,['Metabolism of RNA'],Reactome_2022,1.135994e-07,0.0,0.0,5.896156,94.283001,UTP25;DDX47;DIMT1;SPPL2A;HEATR1;WDR75;WDR43;RR...,NaN,NaN,0.283333,NaN


In [35]:
print(go_df_filtered.to_latex())

\begin{tabular}{lrrllrllrrrrrlllrl}
\toprule
 & Community Index & Community Size & Term & Overlap & Adjusted P-value & Category & Gene_set & P-value & Old P-value & Old Adjusted P-value & Odds Ratio & Combined Score & Genes & GO_ID & Slim_IDs & Overlap (value) & KEGG_ID \\
\midrule
97 & 2 & 1270 & Golgi Membrane (GO:0000139) & 78/427 & 0.000000 & ['cellular anatomical structure'] & GO_Cellular_Component_2023 & 0.000000 & 0.000000 & 0.000000 & 3.446371 & 133.293797 & GALNT12;SCARB2;GPSM1;GALNT11;GALNT14;ST6GALNAC2;KDELR1;PITPNB;ZDHHC5;ZDHHC7;GALNT10;SCAMP5;RAB43;GOLGA5;GLIPR2;DSE;QSOX1;LARGE1;GOLGA7;HS3ST3B1;TMED9;ST6GAL2;COG4;ATG9B;CSGALNACT2;COG2;ZDHHC11;RAB33B;RER1;GORASP2;B3GNT3;B3GNT2;KDELR2;KDELR3;VAMP4;SEC22B;CHST2;B4GALT4;SGMS1;GCNT1;FUT2;FUT4;AGPAT3;SCFD1;FUT8;CHST11;MAN2A2;MAN2A1;LMAN2;MGAT5;PDGFC;CHST15;GCNT3;B4GALNT1;CD59;MGAT1;MGAT2;CYTH1;FKTN;GALNT7;B3GALNT1;GALNT6;GALNT5;GALNT4;ST8SIA1;SURF4;B3GALT4;B3GALT5;B4GAT1;ZDHHC9;DOP1B;MGAT4A;C6ORF89;UST;MGAT4B;ACER3;TRIP11;CHPT1 

In [33]:
list(go_df_filtered["Term"])

['Golgi Membrane (GO:0000139)',
 'Bounding Membrane Of Organelle (GO:0098588)',
 'RNA Binding (GO:0003723)',
 'Protein Glycosylation (GO:0006486)',
 'Metabolism Of RNA R-HSA-8953854',
 'Intracellular Non-Membrane-Bounded Organelle (GO:0043232)',
 'Post-translational Protein Modification R-HSA-597592',
 'Nucleolus (GO:0005730)',
 'Nuclear Lumen (GO:0031981)',
 'Metabolism Of Proteins R-HSA-392499',
 'Protein O-linked Glycosylation (GO:0006493)',
 'Endosomal Transport (GO:0016197)',
 'mRNA Binding (GO:0003729)',
 'Golgi Vesicle Transport (GO:0048193)',
 'RNA transport',
 'Membrane Trafficking R-HSA-199991',
 'N-Glycan biosynthesis',
 'Vesicle-Mediated Transport (GO:0016192)',
 'Endocytic Recycling (GO:0032456)',
 'Endoplasmic Reticulum Membrane (GO:0005789)',
 'Acetylglucosaminyltransferase Activity (GO:0008375)',
 'Vesicle-mediated Transport R-HSA-5653656',
 'Processing Of Capped Intron-Containing Pre-mRNA R-HSA-72203',
 'Intra-Golgi And Retrograde Golgi-to-ER Traffic R-HSA-6811442',
 '

In [34]:
list(go_df_filtered["Category"].unique())

["['cellular anatomical structure']",
 "['binding']",
 '[]',
 "['Metabolism of RNA']",
 "['Metabolism of proteins']",
 "['cellular process']",
 "['localization', 'cellular process']",
 "['Vesicle-mediated transport']",
 "['Glycan biosynthesis and metabolism']",
 "['catalytic activity']",
 "['biological regulation']",
 "['molecular function regulator activity']",
 "['Gene expression (Transcription)']",
 "['Lipid metabolism']",
 "['Signal Transduction']",
 "['Metabolism']",
 "['protein-containing complex']",
 "['translation factor activity']",
 "['localization']",
 "['Disease']",
 "['Chromatin organization']"]